## Features algorithms

### HOG

In [ ]:
def compute_hog_histogram(features: np.ndarray, bin_width: float = 0.1) -> np.ndarray:
    """
    Computes a normalized histogram of HOG feature values.

    Args:
        features: HOG feature vector from skimage.feature.hog
        bin_width: Width of histogram bins (0-1 range)

    Returns:
        Normalized histogram of HOG features with specified binning.
    """
    # Create bins covering full [0, 1] range with specified width
    bin_edges = np.arange(0, 1.0 + bin_width, bin_width)

    # Compute histogram and normalize
    hist, _ = np.histogram(features, bins=bin_edges)
    return hist / hist.sum()  # Return probability distribution

def extract_hog_features(
    image: np.ndarray,
    orientations: int = 9,
    pixels_per_cell: tuple[int, int] = (8, 8),
    cells_per_block: tuple[int, int] = (2, 2),
    **hog_kwargs
) -> np.ndarray:
    """
    Extracts HOG features without resizing images smaller than 16x16.

    Args:
        image: Input image (grayscale or RGB).
        orientations: Number of gradient orientation bins.
        pixels_per_cell: Cell size in pixels.
        cells_per_block: Block size in cells.
        **hog_kwargs: Additional arguments for skimage.feature.hog.

    Returns:
        Normalized HOG feature histogram.

    Raises:
        ValueError: If the image dimensions are smaller than 16x16.
    """
    # Validate input dimensions
    if image.shape[0] < 16 or image.shape[1] < 16:
        raise ValueError("Image dimensions must be at least 16x16 pixels.")

    # Extract HOG features
    try:
        features = hog(
            image,
            orientations=orientations,
            pixels_per_cell=pixels_per_cell,
            cells_per_block=cells_per_block,
            block_norm='L2-Hys',
            channel_axis=-1 if image.ndim == 3 else None,
            **hog_kwargs
        )
    except Exception as e:
        raise RuntimeError(f"HOG feature extraction failed: {str(e)}") from e

    return compute_hog_histogram(features)


### LBP

In [ ]:
def lbp_features_color(img: np.ndarray, radius: int = 1, sampling_pixels: int = 8) -> np.ndarray:
    """
    Computes Local Binary Pattern (LBP) features for a color image.

    Args:
        img: Input image in BGR/RGB format (3 channels).
        radius: Radius of the LBP pattern (default=1).
        sampling_pixels: Number of sampling points in the circular LBP pattern (default=8).

    Returns:
        A concatenated feature vector of LBP histograms for the 3 color channels.

    Raises:
        ValueError: If the input image is empty or does not have 3 color channels.
    """

    # Input validation
    if img.size == 0:
        raise ValueError("The input image is empty.")
    if len(img.shape) != 3 or img.shape[2] != 3:
        raise ValueError("The image must have exactly 3 color channels.")

    # LBP configuration
    METHOD = "uniform"
    BINS = sampling_pixels + 2  # Bin for non-uniform patterns

    # Process each color channel separately
    histograms = []
    for channel_index in range(3):
        # Extract the channel and convert it to the appropriate type
        channel = img[:, :, channel_index].astype(np.uint8)

        # Compute the LBP pattern
        lbp = feature.local_binary_pattern(
            channel,
            P=sampling_pixels,
            R=radius,
            method=METHOD
        )

        # Compute normalized histogram
        hist, _ = np.histogram(
            lbp.ravel(),
            bins=np.arange(0, BINS + 1),
            density=True  # Automatic normalization
        )

        histograms.append(hist)

    # Concatenate histograms and ensure float32 type
    return np.concatenate(histograms).astype(np.float32)

### GLCM

In [ ]:
def extract_glcm_features(image: np.ndarray,
                         distances: list = [1],
                         angles: list = [0, np.pi/4, np.pi/2, 3*np.pi/4]) -> np.ndarray:
    """
    Extracts texture features using Gray-Level Co-occurrence Matrix (GLCM) from a grayscale image.

    Args:
        image: Input RGB image (will be converted to grayscale)
        distances: List of pixel distances for co-occurrence (default: [1])
        angles: List of angles in radians (default: [0°, 45°, 90°, 135°])

    Returns:
        Concatenated feature vector containing:
        [contrast, dissimilarity, homogeneity, energy, correlation]

    Raises:
        ValueError: For invalid image input
    """

    # Input validation
    if not isinstance(image, np.ndarray) or image.size == 0:
        raise ValueError("Input must be a non-empty numpy array")
    if len(image.shape) != 3 or image.shape[2] != 3:
        raise ValueError("Input must be a 3-channel RGB image")

    # Convert to grayscale using OpenCV for better performance
    gray_image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Normalize to 0-255 range if needed
    if gray_image.dtype != np.uint8:
        gray_image = (255 * (gray_image - gray_image.min()) /
                     (gray_image.max() - gray_image.min() + 1e-6)).astype(np.uint8)

    # Calculate GLCM matrix
    glcm = graycomatrix(gray_image,
                       distances=distances,
                       angles=angles,
                       symmetric=True,
                       normed=True)

    # Extract texture properties
    features = []
    for prop in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']:
        feature_values = graycoprops(glcm, prop)
        features.append(feature_values.mean())  # Average across distances/angles

    return np.array(features, dtype=np.float32)

### Label

In [ ]:
class CorrosionLevel(Enum):
    NONE = 'no_corrosion'
    MILD = 'mild_corrosion'
    MODERATE = 'moderate_corrosion'
    SEVERE = 'severe_corrosion'

def classify_corrosion_binary(region: np.ndarray) -> str:
    """
    Classifies image region into corrosion (white) or no corrosion (black) using optimized vector operations.

    Args:
        region: Input image region (3-channel RGB array)

    Returns:
        Classification result ('corrosion' or 'no_corrosion')

    """
    # Validate input
    if not isinstance(region, np.ndarray) or region.size == 0:
        raise ValueError("Invalid input region")

    # Reshape and vectorize operations
    pixels = region.reshape(-1, 3)

    # Count non-black pixels (corrosion)
    corrosion_count = np.any(pixels != [0, 0, 0], axis=1).sum()
    total_pixels = pixels.shape[0]

    return 'corrosion' if corrosion_count / total_pixels > 0.5 else 'no_corrosion'

def classify_corrosion_multiclass(region: np.ndarray) -> Tuple[CorrosionLevel, np.ndarray]:
    """
    Classifies corrosion severity using color-coded pixels with vectorized operations.

    Args:
        region: Input image region (3-channel RGB array)

    Returns:
        Tuple containing:
        - CorrosionLevel enum
        - Normalized color distribution array
    """
    # Validate input
    if not isinstance(region, np.ndarray) or region.size == 0:
        raise ValueError("Invalid input region")

    # Define color thresholds (BGR format - adjust)
    COLOR_THRESHOLDS = {
        CorrosionLevel.NONE: ([0, 0, 0], 10),       # Black
        CorrosionLevel.MILD: ([0, 0, 200], 60),     # Red
        CorrosionLevel.MODERATE: ([0, 200, 0], 60), # Green
        CorrosionLevel.SEVERE: ([0, 200, 200], 60)  # Yellow
    }

    pixels = region.reshape(-1, 3)
    counts = np.zeros(len(COLOR_THRESHOLDS), dtype=int)

    # Vectorized color matching with tolerance
    for idx, (level, (color, tolerance)) in enumerate(COLOR_THRESHOLDS.items()):
        lower_bound = np.array(color) - tolerance
        upper_bound = np.array(color) + tolerance
        counts[idx] = np.sum(np.all((pixels >= lower_bound) & (pixels <= upper_bound), axis=1))

    # Get distribution and classification
    distribution = counts / counts.sum()
    max_idx = np.argmax(counts)

    return list(COLOR_THRESHOLDS.keys())[max_idx], distribution